In [ ]:
!pip install -q "transformers>=4.37.0" accelerate pillow safetensors gradio

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

assert torch.cuda.is_available(), "Please enable GPU: Runtime → Change runtime type → GPU."

device = torch.device("cuda")

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16
).to(device)

blip_model.eval()
print("BLIP model loaded on GPU")


In [ ]:
from PIL import Image
import requests
from io import BytesIO
from IPython.display import display

image_url = "https://images.unsplash.com/photo-1515879218367-8466d910aaa4"

response = requests.get(image_url)
response.raise_for_status()

image = Image.open(BytesIO(response.content)).convert("RGB")
display(image)


In [ ]:
with torch.no_grad():
    inputs = blip_processor(image, return_tensors="pt").to(device, torch.float16)
    output_ids = blip_model.generate(**inputs, max_new_tokens=40)

caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)
print("Generated Caption:", caption)


In [ ]:
import json

# Scene Classifier

def classify_scene_from_caption(caption_text: str) -> str:
    """
    Classify scene based on BLIP caption.
    """
    t = caption_text.lower()

    if any(w in t for w in ["scooter", "e-scooter", "motor scooter"]):
        return "scooter"
    if any(w in t for w in ["construction site", "hard hat", "excavator", "scaffolding", "crane"]):
        return "construction_site"
    if any(w in t for w in ["playground", "slide", "swings"]):
        return "playground"
    if any(w in t for w in ["supermarket", "grocery store", "shopping aisle"]):
        return "supermarket"
    if any(w in t for w in ["street", "sidewalk", "crosswalk", "road"]):
        return "street"
    if any(w in t for w in ["park", "trail", "path"]):
        return "park"
    if any(w in t for w in ["bedroom", "bed", "nightstand"]):
        return "bedroom"
    if any(w in t for w in ["kitchen", "stove", "cooking", "pan", "pot"]):
        return "kitchen"
    if any(w in t for w in ["living room", "sofa", "couch", "coffee table"]):
        return "living_room"
    if any(w in t for w in ["beach", "sand", "shore"]):
        return "beach"
    if any(w in t for w in ["desk", "office", "keyboard", "laptop", "computer"]):
        return "workspace"

    return "generic"


# Hazard Templates

def hazard_template_for_scene(scene: str) -> dict:

    if scene == "scooter":
        return {
            "Safety Hazards": [
                "Loss of balance while riding a scooter may result in falls or injury.",
                "Lack of protective gear can increase the severity of injuries during accidents."
            ],
            "Accessibility Risks": [
                "Shared pathways may create conflicts between scooter users and pedestrians.",
                "Scooters parked improperly may block access routes for some users."
            ],
            "Tripping or Fall Dangers": [
                "Uneven pavement or surface changes may cause sudden instability.",
                "Scooters left on walkways can create unexpected tripping hazards."
            ],
            "Wheelchair Accessibility Issues": [
                "Scooters parked on sidewalks may obstruct wheelchair movement.",
                "Narrow pathways shared with scooters can limit maneuvering space."
            ],
            "Obstacles or Narrow Pathways": [
                "Street furniture and curbs may restrict safe scooter navigation.",
                "Crowded pedestrian areas can create narrow movement paths."
            ],
            "Environmental Conditions": [
                "Road surface conditions may affect traction and stability.",
                "Lighting conditions can impact visibility for scooter users and pedestrians."
            ],
            "Corrective Actions": [
                "Encourage use of helmets and protective equipment while riding.",
                "Designate clear parking and travel zones for scooters."
            ]
        }

    if scene == "beach":
        return {
            "Safety Hazards": [
                "Direct sun exposure may cause overheating or skin discomfort.",
                "Sitting on warm sand for extended periods may contribute to heat buildup."
            ],
            "Accessibility Risks": [
                "Uneven sand may be difficult to navigate for individuals with limited mobility.",
                "Soft sand can make sitting or standing less stable."
            ],
            "Tripping or Fall Dangers": [
                "Uneven sand surfaces may create minor balance challenges.",
                "Foot placement on soft sand may cause instability."
            ],
            "Wheelchair Accessibility Issues": [
                "Soft sand presents difficulty for wheelchair movement.",
                "Lack of firm pathways reduces accessibility."
            ],
            "Obstacles or Narrow Pathways": [
                "Natural sand contours may restrict movement.",
                "Objects placed on sand may reduce maneuvering space."
            ],
            "Environmental Conditions": [
                "Bright sunlight may cause glare or discomfort.",
                "High temperatures may reduce comfort."
            ],
            "Corrective Actions": [
                "Use shade or sunscreen to reduce sun exposure.",
                "Provide stable seating or matting on sand."
            ]
        }

    if scene == "workspace":
        return {
            "Safety Hazards": [
                "Prolonged computer use may cause eye strain or musculoskeletal discomfort.",
                "Loose devices or accessories may pose minor contact hazards."
            ],
            "Accessibility Risks": [
                "Improper placement of devices may limit reach for some users.",
                "Low contrast surfaces may reduce visibility."
            ],
            "Tripping or Fall Dangers": [
                "Cables near desk edges may clutter the floor.",
                "Objects near edges may fall and create tripping hazards."
            ],
            "Wheelchair Accessibility Issues": [
                "Desk height may limit wheelchair access.",
                "Limited leg clearance may restrict positioning."
            ],
            "Obstacles or Narrow Pathways": [
                "Cluttered desk items may restrict arm movement.",
                "Accessories may reduce usable workspace."
            ],
            "Environmental Conditions": [
                "Glare from lighting may affect comfort.",
                "Room temperature or noise may influence usability."
            ],
            "Corrective Actions": [
                "Organize desk layout and manage cables.",
                "Adjust ergonomics and lighting for comfort."
            ]
        }

    # Generic fallback
    return {
        "Safety Hazards": [
            "Prolonged exposure to the environment may cause discomfort.",
            "Unsecured objects may create minor contact hazards."
        ],
        "Accessibility Risks": [
            "Layout may pose challenges for users with limited mobility.",
            "Low visibility may hinder navigation."
        ],
        "Tripping or Fall Dangers": [
            "Irregular surfaces may cause imbalance.",
            "Objects on the ground may contribute to tripping."
        ],
        "Wheelchair Accessibility Issues": [
            "Surface conditions may impede wheelchair movement.",
            "Obstructed areas may reduce maneuverability."
        ],
        "Obstacles or Narrow Pathways": [
            "Objects may limit movement space.",
            "Cluttered areas may reduce clear paths."
        ],
        "Environmental Conditions": [
            "Lighting or temperature may affect comfort.",
            "Environmental variations may influence safety."
        ],
        "Corrective Actions": [
            "Reorganize layout to improve safety.",
            "Provide appropriate accessibility accommodations."
        ]
    }


In [ ]:
scene_type = classify_scene_from_caption(caption)
print("Detected scene type:", scene_type)

final_output = hazard_template_for_scene(scene_type)
print(json.dumps(final_output, indent=2, ensure_ascii=False))


In [ ]:
import gradio as gr

def analyze_image(img):
    # BLIP caption
    with torch.no_grad():
        inputs = blip_processor(img, return_tensors="pt").to(device, torch.float16)
        output_ids = blip_model.generate(**inputs, max_new_tokens=40)
    cap = blip_processor.decode(output_ids[0], skip_special_tokens=True)

    # Scene classification + hazards
    scene = classify_scene_from_caption(cap)
    hazards = hazard_template_for_scene(scene)

    return cap, scene, json.dumps(hazards, indent=2, ensure_ascii=False)

demo = gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=[
        gr.Textbox(label="Generated Caption"),
        gr.Textbox(label="Detected Scene Type"),
        gr.Code(label="Hazard Analysis (JSON)")
    ],
    title="Image-based Hazard & Accessibility Analysis",
    description="BLIP captioning + scene classification + deterministic hazard templates."
)

demo.launch()
